In [ ]:
from IPython.display import HTML
from brax.io import html
import jax
import jax.numpy as jp

from environment import PupperV3Env
# 1. Setup a minimal/empty reward config to satisfy the class
class MockConfig:
    def __init__(self):
        self.scales = {k: 0.0 for k in [
            "tracking_foot_lin_pos", "reward_stand", "tracking_orientation",
            "lin_vel_z", "ang_vel_xy", "orientation", "torques",
            "joint_acceleration", "mechanical_work", "action_rate",
            "abduction_angle", "foot_slip", "termination",
            "knee_collision", "body_collision"
        ]}
        self.tracking_sigma = 0.25

reward_config = type('Config', (), {'rewards': MockConfig()})()

# 2. Initialize the environment
# Make sure your pupper_v3.xml is in the Colab file browser
env = PupperV3Env(
    path="robot_model/pupper_v3.xml", 
    reward_config=reward_config,
    action_scale=0.5,
    observation_history=15
)

# 3. Rollout logic
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

rng = jax.random.PRNGKey(42)
state = reset_fn(rng)
states = []

# Simulate for 100 steps (2 seconds at 0.02s dt)
for _ in range(100):
    states.append(state.pipeline_state)
    # Sending zero actions to observe the 'default_pose'
    state = step_fn(state, jp.zeros(12))

# 4. Render directly in the notebook cell
# We use .replace(vis_static_mocap=True) so your ghost spheres show up
HTML(html.render(env.sys.replace(vis_static_mocap=True), states))